# Porosity Prediction from Seismic Attributes — Machine Learning Workflow with Uncertainty Quantification

A reproducible machine-learning workflow for predicting reservoir **porosity (PHIT)** from
seismic attributes, including model comparison and **uncertainty quantification (UQ)**.

This notebook accompanies my undergraduate thesis:
> **Seismic Attributes Prediction of Porosity Using Machine Learning with Uncertainty
> Quantification: A Case Study of Inas Field, North Malay Basin**
> Thesis: https://repository.its.ac.id/127681/

> ⚠️ **Note on data:** The field data used in the original study (well logs and seismic
> attributes) are **proprietary and are not included here**. This notebook uses **fully
> synthetic data** generated in-code so the methodology is reproducible and self-contained.
> The workflow, models, and evaluation mirror the thesis; only the data is synthetic.

**Workflow**
1. Generate a synthetic multi-well dataset (seismic attributes → porosity)
2. Exploratory analysis & feature correlation
3. Target smoothing and blind-well hold-out
4. Train/test split and feature scaling
5. Gradient-boosting model (XGBoost) with hyperparameter tuning
6. Model comparison (XGBoost, CatBoost, Random Forest, Quantile RF)
7. Uncertainty quantification via bootstrap resampling (*Stochastic XGBoost*)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.utils import resample
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV

import xgboost as xg
from catboost import CatBoostRegressor
from quantile_forest import RandomForestQuantileRegressor

import warnings
warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = "retina"
sns.set_style("whitegrid")

## 1. Synthetic dataset

We synthesise several pseudo-wells. Each well has a set of standard seismic attributes and a
target porosity (`PHIT`) generated as a non-linear function of a few attributes plus noise —
mimicking the structure of the real dataset without exposing any proprietary information.

In [ ]:
rng = np.random.default_rng(42)

ATTRS = ["Instantaneous Amplitude", "Instantaneous Frequency", "RMS Amplitude",
         "Sweetness", "Envelope", "Dominant Frequency", "Curvature", "Chaos"]

def make_well(name, n):
    """Generate one synthetic well: smooth attribute logs + porosity target."""
    t = np.linspace(800, 1200, n)                       # two-way time (ms)
    attrs = {a: rng.normal(0, 1, n).cumsum() / np.sqrt(n) for a in ATTRS}
    df = pd.DataFrame(attrs)
    df.insert(0, "TIME", t)
    df["Well Name"] = name
    # porosity as a non-linear blend of a few attributes + noise
    phi = (0.15
           + 0.05 * np.tanh(df["Instantaneous Amplitude"])
           - 0.03 * df["Instantaneous Frequency"]
           + 0.04 * df["Sweetness"]
           + rng.normal(0, 0.015, n))
    df["PHIT"] = np.clip(phi, 0.01, 0.34)
    return df

wells = pd.concat(
    [make_well(f"Synthetic Well {c}", int(rng.integers(180, 240))) for c in "ABCDE"],
    ignore_index=True,
)
print("Dataset shape:", wells.shape)
wells.head()

In [ ]:
wells.describe().T

## 2. Feature correlation

Inspect how the attributes relate to each other and to porosity. In the original study this
step drove feature selection (dropping redundant / highly collinear attributes).

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(wells[ATTRS + ["PHIT"]].corr(), vmin=-1, vmax=1, annot=True,
            fmt=".2f", cmap="coolwarm", annot_kws={"size": 8})
plt.title("Attribute correlation")
plt.tight_layout()
plt.show()

## 3. Target smoothing & blind-well hold-out

The porosity log is lightly smoothed (Gaussian filter) to suppress high-frequency noise.
One well is held out entirely as a **blind well** to test spatial generalisation — the model
never sees it during training.

In [ ]:
wells["PHIT_smoothed"] = gaussian_filter(wells["PHIT"].values, sigma=2)

blind_well = "Synthetic Well E"
blind = wells[wells["Well Name"] == blind_well].copy()
train = wells[wells["Well Name"] != blind_well].copy()
print(f"Train samples: {train.shape[0]}  |  Blind samples: {blind.shape[0]}")

## 4. Train/test split & scaling

In [ ]:
X = train[ATTRS].values
y = train["PHIT_smoothed"].values
X_blind = blind[ATTRS].values
y_blind = blind["PHIT_smoothed"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
X_blind_s = scaler.transform(X_blind)

X_train_s.shape, X_test_s.shape, X_blind_s.shape

## 5. XGBoost with hyperparameter tuning

We tune an XGBoost regressor with `HalvingRandomSearchCV` (successive halving) over a grid of
tree/regularisation parameters, then evaluate on both the held-out test set and the blind well.

In [ ]:
param_grid = {
    "n_estimators":     [100, 300, 500, 700],
    "max_depth":        [3, 5, 7, 9],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.5, 0.7, 0.9, 1.0],
    "colsample_bytree": [0.5, 0.7, 0.9, 1.0],
    "gamma":            [0, 0.1, 0.2, 0.5],
    "reg_alpha":        [0, 0.1, 0.5, 1],
    "reg_lambda":       [0, 0.1, 0.5, 1],
    "min_child_weight": [1, 3, 5, 7],
}

search = HalvingRandomSearchCV(
    xg.XGBRegressor(objective="reg:squarederror", tree_method="hist"),
    param_grid, factor=3, resource="n_samples", max_resources="auto",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error", n_jobs=-1, random_state=42,
)
search.fit(X_train_s, y_train)
best_params = search.best_params_
print("Best hyperparameters:", best_params)

In [ ]:
def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    pcc  = pearsonr(y_true, y_pred)[0]
    print(f"{name:12s}  RMSE={rmse:.4f}  R2={r2:.4f}  PCC={pcc:.3f}")
    return rmse, r2, pcc

best_xgb = xg.XGBRegressor(objective="reg:squarederror", **best_params)
best_xgb.fit(X_train_s, y_train)

pred_test  = best_xgb.predict(X_test_s)
pred_blind = best_xgb.predict(X_blind_s)

report("Test", y_test, pred_test)
report("Blind", y_blind, pred_blind)

## 6. Model comparison

The thesis benchmarks gradient boosting against tree-based alternatives. Here we compare
XGBoost, CatBoost, Random Forest, and a Quantile Random Forest on the test set.

In [ ]:
models = {
    "XGBoost":      xg.XGBRegressor(objective="reg:squarederror", **best_params),
    "CatBoost":     CatBoostRegressor(verbose=0, iterations=300),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "QuantileRF":   RandomForestQuantileRegressor(n_estimators=300, random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    p = np.ravel(model.predict(X_test_s))
    rows.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, p)),
        "R2":   r2_score(y_test, p),
        "PCC":  pearsonr(y_test, p)[0],
    })

comparison = pd.DataFrame(rows).set_index("Model").sort_values("RMSE")
comparison.style.format("{:.4f}")

## 7. Uncertainty quantification — *Stochastic XGBoost*

Aleatoric/epistemic uncertainty on the blind well is estimated by **bootstrap resampling**:
train many XGBoost models on resampled training sets and use the spread of their predictions
to form a 95% confidence interval (mean ± 1.96·σ).

In [ ]:
n_bootstraps = 100
boot = np.zeros((n_bootstraps, len(X_blind_s)))

for i in range(n_bootstraps):
    X_r, y_r = resample(X_train_s, y_train, random_state=i)
    m = xg.XGBRegressor(objective="reg:squarederror", **best_params, random_state=i)
    m.fit(X_r, y_r)
    boot[i] = m.predict(X_blind_s)

mean_pred = boot.mean(axis=0)
std_pred  = boot.std(axis=0)
lower = mean_pred - 1.96 * std_pred
upper = mean_pred + 1.96 * std_pred
print(f"Mean 95% CI width: {(upper - lower).mean():.4f} (v/v)")

In [ ]:
time = blind["TIME"].values
order = np.argsort(time)

fig, ax = plt.subplots(figsize=(4.5, 9))
ax.plot(y_blind[order], time[order], label="Actual", color="navy", lw=1.5)
ax.plot(mean_pred[order], time[order], label="Predicted (mean)", color="crimson", lw=1.5)
ax.fill_betweenx(time[order], lower[order], upper[order],
                 color="crimson", alpha=0.15, label="95% CI")
ax.invert_yaxis()
ax.set_xlabel("PHIT (v/v)")
ax.set_ylabel("TIME (ms)")
ax.set_title("Blind Well — Stochastic XGBoost")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Summary

This notebook reproduces the porosity-prediction methodology from the thesis on **synthetic
data**: feature correlation, blind-well hold-out, tuned XGBoost regression, multi-model
comparison, and bootstrap-based uncertainty quantification (Stochastic XGBoost).

To apply it to real data, replace the synthetic-data section with your own loader (e.g. `lasio`
for LAS well logs) and adjust the attribute list — the rest of the workflow is data-agnostic.

*Reference:* https://repository.its.ac.id/127681/